# data_make_DRP317_11sandstone_big_dataset_v3_rebuild

本 Notebook 用于重建已误删的：

```text
data_make_DRP317_11sandstone_big_dataset_v3.ipynb
```

目标数据集目录：

```text
H:\Deepl_Project\GNN\ROCK_data\data\DRP317_11sandstone_GNN_dataset_v3
```

输出结构与 PM-GNN 训练 Notebook 对齐：

```text
DRP317_11sandstone_GNN_dataset_v3
├─ raw
├─ pore
├─ Permeation
├─ Mechanics
└─ summary
   ├─ DRP317_11sandstone_blocks_index_v2.csv
   ├─ DRP317_11sandstone_pore_summary_v2.csv
   ├─ DRP317_11sandstone_permeation_summary_v2.csv
   ├─ DRP317_11sandstone_mechanics_summary_v2.csv
   └─ DRP317_11sandstone_GNN_dataset_v2_master.csv
```

## 数据集制作规则

- 原始数据：11个砂岩 `1000×1000×1000` RAW
- 体素：`2.25 μm`
- 裁剪块：`200×200×200`
- 步长：`100`
- 每个样品：`9×9×9 = 729` 个子块
- 11个样品：`8019` 个子样本

## 使用建议

第一次运行建议：

```python
MAX_BLOCKS_PER_ROCK = 10
```

确认流程无误后，再改为：

```python
MAX_BLOCKS_PER_ROCK = None
```

完整生成 8019 个子样本会非常耗时，尤其是 PoreSpy/OpenPNM 网络提取和力学代理计算。


## 重建说明

该 Notebook 以你当前上传的 `data_make_improved_fast_skeleton_no_abnormal_zero.ipynb` 为底稿重建，并做了以下关键修改：

1. 从单个砂岩样品扩展为 11 个砂岩样品批量处理；
2. 将裁剪方式改为 `BLOCK_SIZE=200`、`STRIDE=100`；
3. 每个 `1000³` 样品理论生成 729 个子块；
4. 输出目录改为 `DRP317_11sandstone_GNN_dataset_v3`；
5. 输出总表文件名与 PM-GNN 训练代码匹配；
6. `sample_id` 改为包含 `rock_id`、空间起点和 `b200` 的唯一编号；
7. 保留 `raw / pore / Permeation / Mechanics / summary` 五类输出。


In [ ]:
# ============================================================
# 0. 环境、路径与11砂岩样品自动识别
# ============================================================

from pathlib import Path
import os
import math
import warnings
import random
import re
import shutil
import contextlib
import joblib

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import numpy as np
import pandas as pd
from tqdm import tqdm
from joblib import Parallel, delayed

from scipy import ndimage as ndi
from scipy import sparse
from scipy.sparse.linalg import cg, spsolve

from skimage.morphology import remove_small_objects
from skimage.measure import block_reduce

try:
    from skimage.morphology import skeletonize_3d as _skimage_skeletonize_3d

    def skeletonize_3d_compatible(binary_mask):
        return _skimage_skeletonize_3d(binary_mask).astype(bool)

except Exception:
    from skimage.morphology import skeletonize as _skimage_skeletonize

    def skeletonize_3d_compatible(binary_mask):
        try:
            return _skimage_skeletonize(binary_mask.astype(bool), method="lee").astype(bool)
        except TypeError:
            return _skimage_skeletonize(binary_mask.astype(bool)).astype(bool)

import porespy as ps
import openpnm as op
import networkx as nx

warnings.filterwarnings("ignore")


@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    """
    让 joblib.Parallel 在 Jupyter 中显示 tqdm 进度条。
    """
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_callback
        tqdm_object.close()


# ============================================================
# 需要你确认/修改的路径
# ============================================================

# GitHub portable configuration (only I/O paths changed; modelling code retained).
# Launch Jupyter from the repository root, or set PMGNN_PROJECT_ROOT first.
PROJECT_ROOT = Path(os.environ.get("PMGNN_PROJECT_ROOT", str(Path.cwd()))).expanduser().resolve()
DATA_ROOT = Path(os.environ.get("PMGNN_DATA_ROOT", str(PROJECT_ROOT / "data"))).expanduser().resolve()
DATASET_PREFIX = "DRP317_11sandstone"

BASE_OUT = DATA_ROOT / f"{DATASET_PREFIX}_GNN_dataset_v3"

RAW_OUT = BASE_OUT / "raw"
PORE_OUT = BASE_OUT / "pore"
PERM_OUT = BASE_OUT / "Permeation"
MECH_OUT = BASE_OUT / "Mechanics"
SUMMARY_OUT = BASE_OUT / "summary"

for p in [RAW_OUT, PORE_OUT, PERM_OUT, MECH_OUT, SUMMARY_OUT]:
    p.mkdir(parents=True, exist_ok=True)


def sanitize_rock_id(name: str) -> str:
    name = Path(name).stem
    name = name.replace(" ", "_")
    name = re.sub(r"[^0-9A-Za-z_\-]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name


def discover_rock_samples():
    """
    自动识别DATA_ROOT下11个完整1000³ uint8 RAW砂岩文件。

    会自动排除输出目录中的200³裁剪块。
    如果自动识别结果不正确，请直接手动修改 ROCK_SAMPLES。
    """
    expected_bytes = int(np.prod(SHAPE_ZYX) * np.dtype(DTYPE).itemsize)

    candidates = []
    if DATA_ROOT.exists():
        for path in DATA_ROOT.rglob("*.raw"):
            try:
                if BASE_OUT in path.parents:
                    continue
            except Exception:
                pass

            try:
                if path.stat().st_size != expected_bytes:
                    continue
            except OSError:
                continue

            rock_id = sanitize_rock_id(path.stem)
            candidates.append(
                {
                    "rock_id": rock_id,
                    "rock_name": path.stem,
                    "data_group": "data",
                    "raw_path": path,
                    "shape_zyx": SHAPE_ZYX,
                    "dtype": DTYPE,
                    "voxel_size_um": VOXEL_SIZE_UM,
                }
            )

    candidates = sorted(candidates, key=lambda x: x["rock_id"])
    return candidates


# 若自动识别不准确，可以手动填写，例如：
# ROCK_SAMPLES = [
#     {
#         "rock_id": "BB_2d25um_grayscale_filtered",
#         "rock_name": "BB_2d25um_grayscale_filtered",
#         "data_group": "data",
#         "raw_path": DATA_ROOT / "BB_2d25um_grayscale_filtered.raw",
#         "shape_zyx": SHAPE_ZYX,
#         "dtype": DTYPE,
#         "voxel_size_um": VOXEL_SIZE_UM,
#     },
# ]

print("数据集输出目录：", BASE_OUT)


In [ ]:
# ============================================================
# 1. 全局参数
# ============================================================

SHAPE_ZYX = (1000, 1000, 1000)
DTYPE = np.uint8

VX_UM = 2.25
VY_UM = 2.25
VZ_UM = 2.25
VOXEL_SIZE_UM = 2.25

BLOCK_SIZE = 200
STRIDE = 100

# 第一次测试建议用 10 或 20；正式重建完整数据集时改为 None。
MAX_BLOCKS_PER_ROCK = 10
# MAX_BLOCKS_PER_ROCK = None

PORE_MIN = 0
PORE_MAX = 60
SOLID_MIN = PORE_MAX + 1
SOLID_MAX = 255
MIN_OBJECT_SIZE = 2

ENABLE_SOLID_SKELETON = True
SKELETON_MAX_SIZE = 100
SKELETON_NEIGHBORHOOD = 26

SNOW_R_MAX = 4
SNOW_SIGMA = 0.4
SNOW_BOUNDARY_WIDTH = 3

PIN = 1.0
POUT = 0.0
VISCOSITY = 1.0

MATRIX_E_GPA = 70.0
MATRIX_NU = 0.17
MECH_MAX_SIZE = 50
MECH_SOLID_THRESHOLD = 0.5

CPU_USAGE_RATIO = 0.8
N_JOBS_DATASET = max(1, int((os.cpu_count() or 1) * CPU_USAGE_RATIO))

PARALLEL_PERMEATION = True
N_JOBS_PERMEATION = max(1, min(N_JOBS_DATASET, 12))

PARALLEL_MECHANICS = True
N_JOBS_MECHANICS = max(1, min(N_JOBS_DATASET, 20))

SAVE_INDIVIDUAL_MECH_XLSX = True
SAVE_INDIVIDUAL_PERMEABILITY_XLSX = True

BETWEENNESS_APPROX_K = 200
CENTRALITY_MAX_NODES_EXACT = 300

DIRECTION_AXIS_MASK = {
    "x": 2,
    "y": 1,
    "z": 0,
}

DIRECTION_AXIS_PN = {
    "x": 0,
    "y": 1,
    "z": 2,
}

ROCK_SAMPLES = discover_rock_samples()

print(f"CPU 逻辑线程数: {os.cpu_count()}")
print(f"数据集制作并行任务数: {N_JOBS_DATASET}")
print(f"渗透率并行任务数: {N_JOBS_PERMEATION}")
print(f"力学并行任务数: {N_JOBS_MECHANICS}")
print(f"识别到完整1000³ RAW样品数: {len(ROCK_SAMPLES)}")

if len(ROCK_SAMPLES) != 11:
    print("警告：自动识别到的样品数不是11。请检查 DATA_ROOT 或手动编辑 ROCK_SAMPLES。")

for i, sample in enumerate(ROCK_SAMPLES, start=1):
    print(f"{i:02d}. {sample['rock_id']} -> {sample['raw_path']}")

print("参数设置完成")


## 2. Step 1：RAW 裁剪与空间划分字段

这一部分对应原来的 `raw.py`，但增加了空间分区字段，后续可以用来避免训练集和测试集空间重叠。

In [ ]:
# ============================================================
# 2. Step 1：11砂岩RAW裁剪与空间划分字段
# ============================================================

def check_raw_file(raw_path, shape_zyx, dtype):
    expected_size = int(np.prod(shape_zyx) * np.dtype(dtype).itemsize)
    actual_size = Path(raw_path).stat().st_size
    if actual_size != expected_size:
        raise ValueError(
            f"RAW文件大小不匹配：{raw_path}\n"
            f"期望大小: {expected_size} bytes\n"
            f"实际大小: {actual_size} bytes\n"
            f"请检查 SHAPE_ZYX 或 DTYPE 是否正确。"
        )


def build_positions(shape_zyx, block_size, stride, max_blocks=None):
    z_size, y_size, x_size = shape_zyx
    positions = []

    for z in range(0, z_size - block_size + 1, stride):
        for y in range(0, y_size - block_size + 1, stride):
            for x in range(0, x_size - block_size + 1, stride):
                positions.append((z, y, x))

    if max_blocks is not None:
        positions = positions[:max_blocks]

    return positions


def assign_spatial_split(start_value, axis_size=1000, block_size=200):
    ratio = start_value / max(1, axis_size - block_size)

    if ratio < 0.70:
        return "train"
    elif ratio < 0.85:
        return "val"
    else:
        return "test"


def crop_raw_blocks():
    """
    对11个1000³砂岩RAW统一裁剪。

    输出：
    - raw/*.raw
    - summary/DRP317_11sandstone_blocks_index_v2.csv/xlsx
    """
    if len(ROCK_SAMPLES) == 0:
        raise RuntimeError("ROCK_SAMPLES为空。请检查DATA_ROOT或手动填写ROCK_SAMPLES。")

    index_csv = SUMMARY_OUT / f"{DATASET_PREFIX}_blocks_index_v2.csv"
    index_xlsx = SUMMARY_OUT / f"{DATASET_PREFIX}_blocks_index_v2.xlsx"

    print("=" * 90)
    print("Step 1: 11砂岩RAW裁剪开始")
    print(f"数据根目录: {DATA_ROOT}")
    print(f"样品数量: {len(ROCK_SAMPLES)}")
    print(f"原始尺寸 ZYX: {SHAPE_ZYX}")
    print(f"裁剪尺寸: {BLOCK_SIZE} × {BLOCK_SIZE} × {BLOCK_SIZE}")
    print(f"步长: {STRIDE}")
    print(f"每个样品理论块数: {len(build_positions(SHAPE_ZYX, BLOCK_SIZE, STRIDE, None))}")
    print(f"MAX_BLOCKS_PER_ROCK: {MAX_BLOCKS_PER_ROCK}")
    print(f"raw 输出目录: {RAW_OUT}")
    print("=" * 90)

    rows = []
    global_sample_number = 0

    for rock_index, sample in enumerate(ROCK_SAMPLES):
        rock_id = sample["rock_id"]
        rock_name = sample.get("rock_name", rock_id)
        data_group = sample.get("data_group", "data")
        raw_path = Path(sample["raw_path"])
        shape_zyx = tuple(sample.get("shape_zyx", SHAPE_ZYX))
        dtype = sample.get("dtype", DTYPE)
        voxel_size_um = float(sample.get("voxel_size_um", VOXEL_SIZE_UM))

        check_raw_file(raw_path, shape_zyx, dtype)

        volume = np.memmap(
            raw_path,
            dtype=dtype,
            mode="r",
            shape=shape_zyx,
            order="C",
        )

        positions = build_positions(
            shape_zyx=shape_zyx,
            block_size=BLOCK_SIZE,
            stride=STRIDE,
            max_blocks=MAX_BLOCKS_PER_ROCK,
        )

        print(f"\n[{rock_index + 1}/{len(ROCK_SAMPLES)}] {rock_id}")
        print(f"RAW: {raw_path}")
        print(f"裁剪块数量: {len(positions)}")

        for local_idx, (z, y, x) in enumerate(tqdm(positions, desc=f"Crop {rock_id}", unit="block")):
            sample_id = f"{rock_id}_{local_idx:04d}_z{z}_y{y}_x{x}_b{BLOCK_SIZE}"
            raw_file = RAW_OUT / f"{sample_id}.raw"

            block = np.asarray(
                volume[
                    z:z + BLOCK_SIZE,
                    y:y + BLOCK_SIZE,
                    x:x + BLOCK_SIZE,
                ]
            )

            block.astype(dtype).tofile(raw_file)

            rows.append({
                "sample_number": global_sample_number,
                "rock_index": rock_index,
                "rock_id": rock_id,
                "rock_name": rock_name,
                "data_group": data_group,
                "sample_id": sample_id,
                "local_block_index": local_idx,

                "z_start": z,
                "y_start": y,
                "x_start": x,
                "z_end": z + BLOCK_SIZE,
                "y_end": y + BLOCK_SIZE,
                "x_end": x + BLOCK_SIZE,

                "block_size": BLOCK_SIZE,
                "stride": STRIDE,
                "voxel_size_um": voxel_size_um,
                "raw_file": str(raw_file),
                "source_raw_file": str(raw_path),

                "spatial_group_x": int(x // STRIDE),
                "spatial_group_y": int(y // STRIDE),
                "spatial_group_z": int(z // STRIDE),

                "split_by_x": assign_spatial_split(x, shape_zyx[2], BLOCK_SIZE),
                "split_by_z": assign_spatial_split(z, shape_zyx[0], BLOCK_SIZE),
            })

            global_sample_number += 1

        del volume

        pd.DataFrame(rows).to_csv(index_csv, index=False, encoding="utf-8-sig")

    df = pd.DataFrame(rows)
    df.to_csv(index_csv, index=False, encoding="utf-8-sig")
    df.to_excel(index_xlsx, index=False)

    expected_full = len(ROCK_SAMPLES) * len(build_positions(SHAPE_ZYX, BLOCK_SIZE, STRIDE, None))
    print("=" * 90)
    print("Step 1 完成：RAW裁剪完成")
    print(f"索引 CSV: {index_csv}")
    print(f"索引 Excel: {index_xlsx}")
    print(f"实际样本数: {len(df)}")
    print(f"完整模式理论样本数: {expected_full}")
    print("=" * 90)

    return df


blocks_df = crop_raw_blocks()
blocks_df.head()


## 3. Step 2：孔隙分割与方向性贯通孔提取

这一部分对应原来的 `pore.py`，但核心变化是：

原来只输出一个：

```text
connected_pore.npy
```

现在输出三个方向性贯通孔隙：

```text
connected_pore_x.npy
connected_pore_y.npy
connected_pore_z.npy
```

如果某方向不贯通，该方向 mask 为空，后续该方向渗透率直接记为 0。

In [ ]:
# ============================================================
# 3. 孔隙分割与方向性贯通孔提取
# ============================================================

def threshold_pore(block: np.ndarray) -> np.ndarray:
    pore = (block >= PORE_MIN) & (block <= PORE_MAX)

    if MIN_OBJECT_SIZE is not None and MIN_OBJECT_SIZE > 1:
        pore = remove_small_objects(pore, min_size=MIN_OBJECT_SIZE)

    return pore.astype(bool)


def read_raw_block(raw_file: Path):
    if not raw_file.exists():
        raise FileNotFoundError(f"找不到裁剪 raw 文件: {raw_file}")

    block = np.fromfile(raw_file, dtype=DTYPE)
    expected = BLOCK_SIZE ** 3

    if block.size != expected:
        raise ValueError(
            f"裁剪 raw 文件体素数不匹配: {raw_file}\n"
            f"理论体素数: {expected}\n"
            f"实际体素数: {block.size}\n"
            f"请检查 BLOCK_SIZE 或 raw.py 输出。"
        )

    return block.reshape((BLOCK_SIZE, BLOCK_SIZE, BLOCK_SIZE))


def get_axis_percolating_mask(binary_mask: np.ndarray, axis: int):
    """
    只保留某个方向上从最小面贯通到最大面的连通组。
    如果该方向不贯通，返回全 False mask。
    """
    labels, num = ndi.label(
        binary_mask,
        structure=np.ones((3, 3, 3), dtype=bool)
    )

    if num == 0:
        return np.zeros_like(binary_mask, dtype=bool), False, 0

    face_min = np.take(labels, indices=0, axis=axis)
    face_max = np.take(labels, indices=labels.shape[axis] - 1, axis=axis)

    labs_min = np.unique(face_min)
    labs_max = np.unique(face_max)

    both = np.intersect1d(labs_min, labs_max)
    both = both[both > 0]

    if both.size == 0:
        return np.zeros_like(binary_mask, dtype=bool), False, 0

    mask = np.isin(labels, both)
    return mask.astype(bool), True, int(both.size)


def segment_pores_directional():
    index_csv = SUMMARY_OUT / f"{DATASET_PREFIX}_blocks_index_v2.csv"
    if not index_csv.exists():
        raise FileNotFoundError("请先运行 Step 1：RAW 裁剪。")

    index_df = pd.read_csv(index_csv)

    print("=" * 90)
    print("Step 2: 孔隙分割与方向性贯通孔提取开始")
    print(f"样本数量: {len(index_df)}")
    print("=" * 90)

    summary_rows = []

    summary_csv = SUMMARY_OUT / f"{DATASET_PREFIX}_pore_summary_v2.csv"
    summary_xlsx = SUMMARY_OUT / f"{DATASET_PREFIX}_pore_summary_v2.xlsx"

    for _, row in tqdm(index_df.iterrows(), total=len(index_df)):
        sample_id = row["sample_id"]
        raw_file = Path(row["raw_file"])

        pore_file = PORE_OUT / f"{sample_id}.pore.npy"
        connected_x_file = PORE_OUT / f"{sample_id}.connected_pore_x.npy"
        connected_y_file = PORE_OUT / f"{sample_id}.connected_pore_y.npy"
        connected_z_file = PORE_OUT / f"{sample_id}.connected_pore_z.npy"

        try:
            block = read_raw_block(raw_file)
            pore_mask = threshold_pore(block)

            connected_x, per_x, ncomp_x = get_axis_percolating_mask(
                pore_mask,
                axis=DIRECTION_AXIS_MASK["x"]
            )
            connected_y, per_y, ncomp_y = get_axis_percolating_mask(
                pore_mask,
                axis=DIRECTION_AXIS_MASK["y"]
            )
            connected_z, per_z, ncomp_z = get_axis_percolating_mask(
                pore_mask,
                axis=DIRECTION_AXIS_MASK["z"]
            )

            np.save(pore_file, pore_mask.astype(bool))
            np.save(connected_x_file, connected_x.astype(bool))
            np.save(connected_y_file, connected_y.astype(bool))
            np.save(connected_z_file, connected_z.astype(bool))

            porosity = float(np.mean(pore_mask))
            connected_porosity_x = float(np.mean(connected_x))
            connected_porosity_y = float(np.mean(connected_y))
            connected_porosity_z = float(np.mean(connected_z))

            status = "ok"

        except Exception as e:
            print(f"[PORE ERROR] {sample_id}: {e}")

            porosity = np.nan
            connected_porosity_x = np.nan
            connected_porosity_y = np.nan
            connected_porosity_z = np.nan
            per_x = per_y = per_z = False
            ncomp_x = ncomp_y = ncomp_z = 0
            status = f"failed: {e}"

        summary_rows.append({
            **row.to_dict(),
            "porosity": porosity,

            "percolates_x": int(per_x),
            "percolates_y": int(per_y),
            "percolates_z": int(per_z),

            "num_percolating_components_x": ncomp_x,
            "num_percolating_components_y": ncomp_y,
            "num_percolating_components_z": ncomp_z,

            "connected_porosity_x": connected_porosity_x,
            "connected_porosity_y": connected_porosity_y,
            "connected_porosity_z": connected_porosity_z,

            # 兼容旧字段，取三个方向连通孔隙率的最大值
            "connected_porosity": np.nanmax([
                connected_porosity_x,
                connected_porosity_y,
                connected_porosity_z
            ]),

            "pore_file": str(pore_file),
            "connected_pore_x_file": str(connected_x_file),
            "connected_pore_y_file": str(connected_y_file),
            "connected_pore_z_file": str(connected_z_file),
            "pore_status": status
        })

        pd.DataFrame(summary_rows).to_csv(
            summary_csv,
            index=False,
            encoding="utf-8-sig"
        )

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")
    summary_df.to_excel(summary_xlsx, index=False)

    print("=" * 90)
    print("Step 2 完成：孔隙分割与方向性贯通孔提取完成")
    print(f"汇总 CSV: {summary_csv}")
    print(f"汇总 Excel: {summary_xlsx}")
    print("=" * 90)

    return summary_df


# 运行 Step 2
pore_summary_df = segment_pores_directional()
pore_summary_df.head()


## 4. Step 3：PNM 网络提取、方向性渗透率计算与图拓扑特征

这一部分对应原来的 `Permeation.py`，但做了三类关键修改：

1. `kx` 只使用 X 方向贯通孔隙；
2. `ky` 只使用 Y 方向贯通孔隙；
3. `kz` 只使用 Z 方向贯通孔隙。

如果某个方向不贯通，则该方向渗透率为 0。

同时保存方向性 PNM 表，并计算图级拓扑特征。

In [ ]:
# ============================================================
# 4. PNM 网络、方向性渗透率与图特征
# ============================================================

PORE_COLUMNS = [
    "Pore ID",
    "Volume [cm^3]",
    "Area [cm^2]",
    "EqRadius [cm]",
    "LabelID",
    "Pressure",
    "X Coord [cm]",
    "Y Coord [cm]",
    "Z Coord [cm]",
    "Coordination Number",
]

THROAT_COLUMNS = [
    "Throat ID",
    "Area [cm^2]",
    "EqRadius [cm]",
    "ChannelLength [cm]",
    "FlowRatePerSec [cm^3]",
    "Pore ID #1",
    "Pore ID #2",
]


def safe_get(obj, keys, default=None):
    for key in keys:
        try:
            if key in obj.keys():
                return np.asarray(obj[key])
        except Exception:
            pass

        try:
            return np.asarray(obj[key])
        except Exception:
            pass

    return default


def get_pore_throat_arrays(pn):
    coords = np.asarray(pn["pore.coords"], dtype=float)
    conns = np.asarray(pn["throat.conns"], dtype=int)
    return coords, conns


def extract_network_by_porespy(mask: np.ndarray):
    if mask.sum() == 0:
        return None, None

    phases = mask.astype(np.int32)

    try:
        snow = ps.networks.snow2(
            phases=phases,
            voxel_size=VOXEL_SIZE_UM,
            boundary_width=SNOW_BOUNDARY_WIDTH,
            sigma=SNOW_SIGMA,
            r_max=SNOW_R_MAX,
        )
    except TypeError:
        snow = ps.networks.snow2(
            phases=phases,
            voxel_size=VOXEL_SIZE_UM,
        )

    net = snow.network

    try:
        regions = snow.regions
    except Exception:
        regions = None

    return net, regions


def network_to_openpnm(net):
    try:
        pn = op.io.network_from_porespy(net)
        return pn
    except Exception:
        pass

    coords = np.asarray(net["pore.coords"], dtype=float)
    conns = np.asarray(net["throat.conns"], dtype=int)

    try:
        pn = op.network.Network(coords=coords, conns=conns)
    except Exception:
        pn = op.network.GenericNetwork()
        pn["pore.coords"] = coords
        pn["throat.conns"] = conns

    for key, value in net.items():
        try:
            pn[key] = value
        except Exception:
            pass

    return pn


def build_pnm_excel_tables(pn, net, origin_zyx):
    coords_um, conns = get_pore_throat_arrays(pn)

    n_pores = coords_um.shape[0]
    n_throats = conns.shape[0]

    z0, y0, x0 = origin_zyx

    coords_global_um = coords_um.copy()
    coords_global_um[:, 0] += x0 * VOXEL_SIZE_UM
    coords_global_um[:, 1] += y0 * VOXEL_SIZE_UM
    coords_global_um[:, 2] += z0 * VOXEL_SIZE_UM

    pore_diameter_um = safe_get(
        pn,
        ["pore.diameter", "pore.equivalent_diameter", "pore.inscribed_diameter"],
        default=None
    )
    if pore_diameter_um is None:
        pore_diameter_um = safe_get(
            net,
            ["pore.diameter", "pore.equivalent_diameter", "pore.inscribed_diameter"],
            default=np.ones(n_pores) * VOXEL_SIZE_UM
        )

    pore_volume_um3 = safe_get(pn, ["pore.volume"], default=None)
    if pore_volume_um3 is None:
        pore_volume_um3 = safe_get(
            net,
            ["pore.volume"],
            default=(4.0 / 3.0) * np.pi * (pore_diameter_um / 2.0) ** 3
        )

    pore_area_um2 = safe_get(
        pn,
        ["pore.surface_area", "pore.area"],
        default=None
    )
    if pore_area_um2 is None:
        pore_area_um2 = safe_get(
            net,
            ["pore.surface_area", "pore.area"],
            default=4.0 * np.pi * (pore_diameter_um / 2.0) ** 2
        )

    pore_label = safe_get(
        pn,
        ["pore.region_label", "pore.label"],
        default=None
    )
    if pore_label is None:
        pore_label = safe_get(
            net,
            ["pore.region_label", "pore.label"],
            default=np.arange(n_pores)
        )

    coordination = np.zeros(n_pores, dtype=int)
    for p1, p2 in conns:
        if 0 <= p1 < n_pores:
            coordination[p1] += 1
        if 0 <= p2 < n_pores:
            coordination[p2] += 1

    coords_cm = coords_global_um * 1e-4
    pore_volume_cm3 = pore_volume_um3 * 1e-12
    pore_area_cm2 = pore_area_um2 * 1e-8
    pore_eq_radius_cm = (pore_diameter_um / 2.0) * 1e-4

    pore_rows = []

    for i in range(n_pores):
        pore_rows.append([
            i,
            float(pore_volume_cm3[i]),
            float(pore_area_cm2[i]),
            float(pore_eq_radius_cm[i]),
            int(pore_label[i]) if i < len(pore_label) else i,
            130000,
            float(coords_cm[i, 0]),
            float(coords_cm[i, 1]),
            float(coords_cm[i, 2]),
            int(coordination[i]),
        ])

    pore_df = pd.DataFrame(pore_rows, columns=PORE_COLUMNS)

    throat_diameter_um = safe_get(
        pn,
        ["throat.diameter", "throat.equivalent_diameter", "throat.inscribed_diameter"],
        default=None
    )
    if throat_diameter_um is None:
        throat_diameter_um = safe_get(
            net,
            ["throat.diameter", "throat.equivalent_diameter", "throat.inscribed_diameter"],
            default=np.ones(n_throats) * VOXEL_SIZE_UM
        )

    throat_area_um2 = safe_get(
        pn,
        ["throat.cross_sectional_area", "throat.area"],
        default=None
    )
    if throat_area_um2 is None:
        throat_area_um2 = safe_get(
            net,
            ["throat.cross_sectional_area", "throat.area"],
            default=np.pi * (throat_diameter_um / 2.0) ** 2
        )

    throat_length_um = safe_get(
        pn,
        ["throat.total_length", "throat.direct_length", "throat.length"],
        default=None
    )
    if throat_length_um is None:
        throat_length_um = safe_get(
            net,
            ["throat.total_length", "throat.direct_length", "throat.length"],
            default=None
        )

    if throat_length_um is None:
        c1 = coords_um[conns[:, 0]]
        c2 = coords_um[conns[:, 1]]
        throat_length_um = np.linalg.norm(c1 - c2, axis=1)

    throat_area_cm2 = throat_area_um2 * 1e-8
    throat_eq_radius_cm = (throat_diameter_um / 2.0) * 1e-4
    throat_length_cm = throat_length_um * 1e-4

    throat_rows = []

    for i in range(n_throats):
        throat_rows.append([
            i,
            float(throat_area_cm2[i]),
            float(throat_eq_radius_cm[i]),
            float(throat_length_cm[i]),
            0.0,
            int(conns[i, 0]),
            int(conns[i, 1]),
        ])

    throat_df = pd.DataFrame(throat_rows, columns=THROAT_COLUMNS)

    return pore_df, throat_df


def write_pnm_xlsx(pore_df, throat_df, out_path: Path):
    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        pore_df.to_excel(writer, sheet_name="Pores", index=False)
        throat_df.to_excel(writer, sheet_name="Throats", index=False)
        pd.DataFrame().to_excel(writer, sheet_name="Sheet3", index=False, header=False)


def compute_hydraulic_conductance(pn):
    coords_um, conns = get_pore_throat_arrays(pn)
    n_throats = conns.shape[0]

    throat_diameter_um = safe_get(
        pn,
        ["throat.diameter", "throat.equivalent_diameter", "throat.inscribed_diameter"],
        default=None
    )

    if throat_diameter_um is None:
        throat_area_um2 = safe_get(
            pn,
            ["throat.cross_sectional_area", "throat.area"],
            default=None
        )

        if throat_area_um2 is not None:
            throat_diameter_um = 2.0 * np.sqrt(throat_area_um2 / np.pi)
        else:
            throat_diameter_um = np.ones(n_throats) * VOXEL_SIZE_UM

    throat_length_um = safe_get(
        pn,
        ["throat.total_length", "throat.direct_length", "throat.length"],
        default=None
    )

    if throat_length_um is None:
        c1 = coords_um[conns[:, 0]]
        c2 = coords_um[conns[:, 1]]
        throat_length_um = np.linalg.norm(c1 - c2, axis=1)

    throat_length_um = np.maximum(throat_length_um, VOXEL_SIZE_UM)
    r_um = np.maximum(throat_diameter_um / 2.0, VOXEL_SIZE_UM / 10.0)

    g = np.pi * r_um ** 4 / (8.0 * VISCOSITY * throat_length_um)
    return g


def get_boundary_pores_from_coords(pn, axis_index, side="min"):
    coords = np.asarray(pn["pore.coords"], dtype=float)
    vals = coords[:, axis_index]

    block_len_um = BLOCK_SIZE * VOXEL_SIZE_UM

    # 方向性贯通网络理论上应该接近 0 和 block_len_um
    # 这里使用更严格的边界容差
    tol = 2.0 * VOXEL_SIZE_UM

    if side == "min":
        pores = np.where(vals <= tol)[0]
    else:
        pores = np.where(vals >= block_len_um - tol)[0]

    # 如果因为 PoreSpy 坐标定义导致找不到边界孔，再退一步使用 min/max 坐标
    if len(pores) == 0:
        vmin = np.min(vals)
        vmax = np.max(vals)
        if side == "min":
            pores = np.where(vals <= vmin + tol)[0]
        else:
            pores = np.where(vals >= vmax - tol)[0]

    return pores



def calculate_absolute_permeability_directional(pn, direction):
    """
    使用孔隙网络图的稀疏拉普拉斯方程直接计算方向性渗透率。

    这一版不再把求解失败直接写成 0：
    - 真实不贯通由上游 percolates_x/y/z 判定，并在 process_permeation_row 中写 0；
    - 只要当前 PNM 求解失败、边界孔缺失、图中没有 inlet-outlet 有效连通分量，
      这里返回 np.nan 和明确 status，避免把异常样本误当作真实 0。
    """
    diag = {
        "n_pores_used": 0,
        "n_throats_used": 0,
        "n_inlet_pores": 0,
        "n_outlet_pores": 0,
        "flow_rate": np.nan,
    }

    if pn is None or "throat.conns" not in pn.keys() or "pore.coords" not in pn.keys():
        return np.nan, "invalid_network", diag

    conns = np.asarray(pn["throat.conns"], dtype=np.int64)
    coords = np.asarray(pn["pore.coords"], dtype=float)

    if conns.size == 0 or coords.size == 0:
        return np.nan, "empty_network", diag

    n_pores = coords.shape[0]
    axis_index = DIRECTION_AXIS_PN[direction]

    inlets = np.asarray(get_boundary_pores_from_coords(pn, axis_index, side="min"), dtype=np.int64)
    outlets = np.asarray(get_boundary_pores_from_coords(pn, axis_index, side="max"), dtype=np.int64)

    inlets = inlets[(inlets >= 0) & (inlets < n_pores)]
    outlets = outlets[(outlets >= 0) & (outlets < n_pores)]

    diag["n_inlet_pores"] = int(len(inlets))
    diag["n_outlet_pores"] = int(len(outlets))

    if len(inlets) == 0 or len(outlets) == 0:
        return np.nan, "boundary_pores_missing", diag

    try:
        g_all = np.asarray(compute_hydraulic_conductance(pn), dtype=np.float64)
    except Exception as e:
        return np.nan, f"conductance_failed: {e}", diag

    valid_edges = (
        (conns[:, 0] >= 0) & (conns[:, 0] < n_pores) &
        (conns[:, 1] >= 0) & (conns[:, 1] < n_pores) &
        np.isfinite(g_all) & (g_all > 0)
    )

    conns = conns[valid_edges]
    g_all = g_all[valid_edges]

    if conns.shape[0] == 0:
        return np.nan, "no_valid_throats", diag

    # 先用 networkx 找到同时包含 inlet 和 outlet 的有效连通分量。
    # 这样可以避免孤立孔簇导致线性方程奇异。
    try:
        G = nx.Graph()
        G.add_nodes_from(range(n_pores))
        G.add_edges_from((int(a), int(b)) for a, b in conns)

        inlet_set = set(int(i) for i in inlets)
        outlet_set = set(int(o) for o in outlets)

        active_nodes = set()
        for comp in nx.connected_components(G):
            if comp & inlet_set and comp & outlet_set:
                active_nodes.update(comp)

        if len(active_nodes) == 0:
            return np.nan, "network_not_connected_between_boundaries", diag

        active_nodes = np.array(sorted(active_nodes), dtype=np.int64)
    except Exception as e:
        return np.nan, f"connectivity_check_failed: {e}", diag

    old_to_new = -np.ones(n_pores, dtype=np.int64)
    old_to_new[active_nodes] = np.arange(len(active_nodes), dtype=np.int64)

    edge_mask = (old_to_new[conns[:, 0]] >= 0) & (old_to_new[conns[:, 1]] >= 0)
    conns_active_old = conns[edge_mask]
    g = g_all[edge_mask]

    if conns_active_old.shape[0] == 0:
        return np.nan, "no_active_throats", diag

    conns_active = np.column_stack([
        old_to_new[conns_active_old[:, 0]],
        old_to_new[conns_active_old[:, 1]],
    ]).astype(np.int64)

    n = len(active_nodes)
    diag["n_pores_used"] = int(n)
    diag["n_throats_used"] = int(conns_active.shape[0])

    inlet_active = np.isin(active_nodes, inlets)
    outlet_active = np.isin(active_nodes, outlets)
    fixed = inlet_active | outlet_active

    if inlet_active.sum() == 0 or outlet_active.sum() == 0:
        return np.nan, "active_boundary_pores_missing", diag

    p_fixed = np.zeros(n, dtype=np.float64)
    p_fixed[inlet_active] = PIN
    p_fixed[outlet_active] = POUT

    rows = []
    cols = []
    data = []

    for (i, j), gij in zip(conns_active, g):
        i = int(i)
        j = int(j)
        gij = float(gij)

        rows.extend([i, i, j, j])
        cols.extend([i, j, i, j])
        data.extend([gij, -gij, -gij, gij])

    Lmat = sparse.csr_matrix((data, (rows, cols)), shape=(n, n))

    free = ~fixed
    p = p_fixed.copy()

    try:
        if np.any(free):
            A_ff = Lmat[free][:, free]
            b_f = -Lmat[free][:, fixed].dot(p_fixed[fixed])

            try:
                p_free, info = cg(A_ff, b_f, rtol=1e-8, maxiter=3000)
                if info != 0 or not np.all(np.isfinite(p_free)):
                    p_free = spsolve(A_ff, b_f)
            except Exception:
                p_free = spsolve(A_ff, b_f)

            if not np.all(np.isfinite(p_free)):
                return np.nan, "pressure_solution_nonfinite", diag

            p[free] = p_free
    except Exception as e:
        return np.nan, f"linear_solve_failed: {e}", diag

    # 总流量：统计 inlet 节点流向非 inlet 节点的通量，避免只看轴向相邻边导致误判为 0。
    Q = 0.0
    for (i, j), gij in zip(conns_active, g):
        i = int(i)
        j = int(j)
        gij = float(gij)

        if inlet_active[i] and not inlet_active[j]:
            Q += gij * max(p[i] - p[j], 0.0)
        elif inlet_active[j] and not inlet_active[i]:
            Q += gij * max(p[j] - p[i], 0.0)

    diag["flow_rate"] = float(Q)

    if Q <= 0 or not np.isfinite(Q):
        return np.nan, "zero_or_nonfinite_flow", diag

    L = np.max(coords[active_nodes, axis_index]) - np.min(coords[active_nodes, axis_index])
    L = max(float(L), VOXEL_SIZE_UM)

    side_um = BLOCK_SIZE * VOXEL_SIZE_UM
    A = side_um * side_um
    delta_p = PIN - POUT

    k_um2 = Q * VISCOSITY * L / (A * delta_p)

    if not np.isfinite(k_um2) or k_um2 <= 0:
        return np.nan, "invalid_permeability", diag

    return float(k_um2), "ok", diag


def graph_features_from_pnm_tables(pore_df, throat_df):
    """
    从 PNM 表计算图级拓扑特征。
    """
    features = {}

    n_pores = len(pore_df)
    n_throats = len(throat_df)

    features["num_pores"] = n_pores
    features["num_throats"] = n_throats

    if n_pores == 0:
        return {
            **features,
            "avg_pore_radius_cm": np.nan,
            "std_pore_radius_cm": np.nan,
            "avg_throat_radius_cm": np.nan,
            "std_throat_radius_cm": np.nan,
            "avg_throat_length_cm": np.nan,
            "std_throat_length_cm": np.nan,
            "avg_coordination_number": np.nan,
            "max_coordination_number": np.nan,
            "network_density": np.nan,
            "avg_betweenness": np.nan,
            "max_betweenness": np.nan,
            "avg_closeness": np.nan,
            "avg_pagerank": np.nan,
            "max_pagerank": np.nan,
        }

    pore_radius = pd.to_numeric(pore_df["EqRadius [cm]"], errors="coerce")
    coord_num = pd.to_numeric(pore_df["Coordination Number"], errors="coerce")

    features["avg_pore_radius_cm"] = float(pore_radius.mean())
    features["std_pore_radius_cm"] = float(pore_radius.std())
    features["avg_coordination_number"] = float(coord_num.mean())
    features["max_coordination_number"] = float(coord_num.max())

    if n_throats > 0:
        throat_radius = pd.to_numeric(throat_df["EqRadius [cm]"], errors="coerce")
        throat_length = pd.to_numeric(throat_df["ChannelLength [cm]"], errors="coerce")

        features["avg_throat_radius_cm"] = float(throat_radius.mean())
        features["std_throat_radius_cm"] = float(throat_radius.std())
        features["avg_throat_length_cm"] = float(throat_length.mean())
        features["std_throat_length_cm"] = float(throat_length.std())
    else:
        features["avg_throat_radius_cm"] = np.nan
        features["std_throat_radius_cm"] = np.nan
        features["avg_throat_length_cm"] = np.nan
        features["std_throat_length_cm"] = np.nan

    try:
        G = nx.Graph()
        G.add_nodes_from(range(n_pores))

        if n_throats > 0:
            edges = []
            for _, r in throat_df.iterrows():
                p1 = int(r["Pore ID #1"])
                p2 = int(r["Pore ID #2"])
                length = float(r["ChannelLength [cm]"])
                radius = float(r["EqRadius [cm]"])
                if p1 != p2 and p1 >= 0 and p2 >= 0 and p1 < n_pores and p2 < n_pores:
                    # 距离权重用长度，流动能力权重可用半径
                    edges.append((p1, p2, {"length": max(length, 1e-30), "radius": radius}))
            G.add_edges_from(edges)

        features["network_density"] = float(nx.density(G))

        if G.number_of_edges() > 0 and G.number_of_nodes() > 1:
            if G.number_of_nodes() <= CENTRALITY_MAX_NODES_EXACT:
                bet = nx.betweenness_centrality(G, weight="length", normalized=True)
            else:
                k = min(BETWEENNESS_APPROX_K, G.number_of_nodes())
                bet = nx.betweenness_centrality(
                    G,
                    k=k,
                    weight="length",
                    normalized=True,
                    seed=42
                )

            clo = nx.closeness_centrality(G, distance="length")
            pr = nx.pagerank(G, weight="radius")

            bet_values = np.array(list(bet.values()), dtype=float)
            clo_values = np.array(list(clo.values()), dtype=float)
            pr_values = np.array(list(pr.values()), dtype=float)

            features["avg_betweenness"] = float(np.mean(bet_values))
            features["max_betweenness"] = float(np.max(bet_values))
            features["avg_closeness"] = float(np.mean(clo_values))
            features["avg_pagerank"] = float(np.mean(pr_values))
            features["max_pagerank"] = float(np.max(pr_values))
        else:
            features["avg_betweenness"] = 0.0
            features["max_betweenness"] = 0.0
            features["avg_closeness"] = 0.0
            features["avg_pagerank"] = 0.0
            features["max_pagerank"] = 0.0

    except Exception as e:
        features["network_density"] = np.nan
        features["avg_betweenness"] = np.nan
        features["max_betweenness"] = np.nan
        features["avg_closeness"] = np.nan
        features["avg_pagerank"] = np.nan
        features["max_pagerank"] = np.nan
        features["graph_feature_error"] = str(e)

    return features


In [ ]:
# ============================================================
# 4.1 运行方向性渗透率计算
# ============================================================

def write_directional_permeability_xlsx(sample_id, row_result, out_path: Path):
    """
    写入 Permeation 文件夹中的逐样本方向性渗透率结果。
    这样 Permeation 文件夹不再为空，且每个样本都有独立结果文件。
    """
    rows = []
    for d in ["x", "y", "z"]:
        rows.append({
            "sample_id": sample_id,
            "direction": d.upper(),
            "percolates": int(row_result.get(f"percolates_{d}", 0)),
            "permeability_um2": float(row_result.get(f"k{d}_um2", 0.0)),
            "status": row_result.get(f"k{d}_status", ""),
            "pnm_xlsx": row_result.get(f"pnm_{d}_xlsx", ""),
        })

    df = pd.DataFrame(rows)

    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name="Directional_Permeability", index=False)


def process_permeation_row(row_dict):
    """
    单个样本的方向性渗透率计算函数。
    设计成独立函数，方便 joblib 并行。
    """
    row = pd.Series(row_dict)

    sample_id = row["sample_id"]
    origin_zyx = (
        int(row.get("z_start", 0)),
        int(row.get("y_start", 0)),
        int(row.get("x_start", 0)),
    )

    result = {
        "sample_id": sample_id,
        "z_start": row.get("z_start", np.nan),
        "y_start": row.get("y_start", np.nan),
        "x_start": row.get("x_start", np.nan),
        "block_size": row.get("block_size", BLOCK_SIZE),

        "percolates_x": int(row.get("percolates_x", 0)),
        "percolates_y": int(row.get("percolates_y", 0)),
        "percolates_z": int(row.get("percolates_z", 0)),

        "kx_um2": np.nan,
        "ky_um2": np.nan,
        "kz_um2": np.nan,

        "kx_status": "not_percolating",
        "ky_status": "not_percolating",
        "kz_status": "not_percolating",

        "permeation_status": "ok",
    }

    permeability_xlsx = PERM_OUT / f"{sample_id}.Directional_KTensor_v2.xlsx"
    result["permeability_xlsx"] = str(permeability_xlsx)

    graph_feature_rows = []

    try:
        for direction in ["x", "y", "z"]:
            per_flag = int(row.get(f"percolates_{direction}", 0))
            connected_file = Path(row[f"connected_pore_{direction}_file"])

            # PNM 图仍保存在 pore 文件夹，因为它是孔隙网络结构；
            # Permeation 文件夹保存方向性渗透率结果表。
            pnm_xlsx = PORE_OUT / f"{sample_id}.PNM_{direction}.flooded.am.xlsx"
            result[f"pnm_{direction}_xlsx"] = str(pnm_xlsx)

            if per_flag == 0:
                pore_df = pd.DataFrame(columns=PORE_COLUMNS)
                throat_df = pd.DataFrame(columns=THROAT_COLUMNS)
                write_pnm_xlsx(pore_df, throat_df, pnm_xlsx)
                result[f"k{direction}_um2"] = 0.0
                result[f"k{direction}_status"] = "not_percolating"
                continue

            mask = np.load(connected_file).astype(bool)

            if mask.sum() == 0:
                pore_df = pd.DataFrame(columns=PORE_COLUMNS)
                throat_df = pd.DataFrame(columns=THROAT_COLUMNS)
                write_pnm_xlsx(pore_df, throat_df, pnm_xlsx)
                result[f"k{direction}_um2"] = np.nan
                result[f"k{direction}_status"] = "empty_mask"
                continue

            net, regions = extract_network_by_porespy(mask)
            pn = network_to_openpnm(net)

            pore_df, throat_df = build_pnm_excel_tables(
                pn,
                net,
                origin_zyx=origin_zyx
            )
            write_pnm_xlsx(pore_df, throat_df, pnm_xlsx)

            k_val, k_status, k_diag = calculate_absolute_permeability_directional(pn, direction)
            result[f"k{direction}_um2"] = k_val
            result[f"k{direction}_status"] = k_status
            result[f"k{direction}_n_pores_used"] = k_diag.get("n_pores_used", np.nan)
            result[f"k{direction}_n_throats_used"] = k_diag.get("n_throats_used", np.nan)
            result[f"k{direction}_n_inlet_pores"] = k_diag.get("n_inlet_pores", np.nan)
            result[f"k{direction}_n_outlet_pores"] = k_diag.get("n_outlet_pores", np.nan)
            result[f"k{direction}_flow_rate"] = k_diag.get("flow_rate", np.nan)

            gf = graph_features_from_pnm_tables(pore_df, throat_df)
            gf = {f"{direction}_{k}": v for k, v in gf.items()}
            graph_feature_rows.append(gf)

        # 合并三个方向的图特征
        for gf in graph_feature_rows:
            result.update(gf)

        # 为了兼容旧流程，额外给一个主图特征：
        # 优先使用 x 方向，没有则使用 y，再没有则使用 z
        for direction in ["x", "y", "z"]:
            if int(row.get(f"percolates_{direction}", 0)) == 1:
                for key in [
                    "num_pores", "num_throats",
                    "avg_pore_radius_cm", "std_pore_radius_cm",
                    "avg_throat_radius_cm", "std_throat_radius_cm",
                    "avg_throat_length_cm", "std_throat_length_cm",
                    "avg_coordination_number", "max_coordination_number",
                    "network_density", "avg_betweenness", "max_betweenness",
                    "avg_closeness", "avg_pagerank", "max_pagerank"
                ]:
                    dkey = f"{direction}_{key}"
                    if dkey in result:
                        result[key] = result[dkey]
                break

        if SAVE_INDIVIDUAL_PERMEABILITY_XLSX:
            write_directional_permeability_xlsx(sample_id, result, permeability_xlsx)

    except Exception as e:
        result["permeation_status"] = f"failed: {e}"
        try:
            if SAVE_INDIVIDUAL_PERMEABILITY_XLSX:
                write_directional_permeability_xlsx(sample_id, result, permeability_xlsx)
        except Exception:
            pass

    return result


def compute_directional_permeability_dataset():
    pore_summary_csv = SUMMARY_OUT / f"{DATASET_PREFIX}_pore_summary_v2.csv"
    if not pore_summary_csv.exists():
        raise FileNotFoundError("请先运行 Step 2：孔隙分割与方向性贯通孔提取。")

    df = pd.read_csv(pore_summary_csv)

    print("=" * 90)
    print("Step 3: 方向性 PNM 网络提取、渗透率计算与图特征提取开始")
    print(f"样本数量: {len(df)}")
    print(f"Permeation 输出目录: {PERM_OUT}")
    print(f"是否并行: {PARALLEL_PERMEATION}, 并行任务数: {N_JOBS_PERMEATION}")
    print("=" * 90)

    rows = df.to_dict("records")

    if PARALLEL_PERMEATION and len(rows) > 1:
        with tqdm_joblib(tqdm(total=len(rows), desc="Step 3 方向性渗透率计算进度", unit="样本")):
            summary_rows = Parallel(
                n_jobs=N_JOBS_PERMEATION,
                backend="loky",
                batch_size=1,
                verbose=0
            )(
                delayed(process_permeation_row)(r)
                for r in rows
            )
    else:
        summary_rows = []
        for r in tqdm(rows, desc="Step 3 方向性渗透率计算进度", unit="样本"):
            summary_rows.append(process_permeation_row(r))

    summary_df = pd.DataFrame(summary_rows)

    summary_csv = SUMMARY_OUT / f"{DATASET_PREFIX}_permeation_summary_v2.csv"
    summary_xlsx = SUMMARY_OUT / f"{DATASET_PREFIX}_permeation_summary_v2.xlsx"

    # summary 目录保存总表
    summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")
    summary_df.to_excel(summary_xlsx, index=False)

    # Permeation 目录也保存一份总表，方便查看，不再出现空文件夹
    perm_summary_csv = PERM_OUT / f"{DATASET_PREFIX}_permeation_summary_v2.csv"
    perm_summary_xlsx = PERM_OUT / f"{DATASET_PREFIX}_permeation_summary_v2.xlsx"
    summary_df.to_csv(perm_summary_csv, index=False, encoding="utf-8-sig")
    summary_df.to_excel(perm_summary_xlsx, index=False)

    print("=" * 90)
    print("Step 3 完成：方向性渗透率与图特征计算完成")
    print(f"汇总 CSV: {summary_csv}")
    print(f"汇总 Excel: {summary_xlsx}")
    print(f"Permeation 文件夹总表: {perm_summary_csv}")
    print("=" * 90)

    return summary_df


# 运行 Step 3
permeation_summary_df = compute_directional_permeability_dataset()
permeation_summary_df.head()


## 5. Step 4：力学代理标签、固体相连通特征与三维固体骨架特征

这一部分对应原来的 `Mechanics.py`，但进一步明确区分三类信息：

1. **固体相阈值分割**：由原始灰度块直接得到 `solid_mask = block > PORE_MAX`，即 19–255 为固体相；
2. **固体相方向性连通特征**：`solid_percolates_x/y/z`、`solid_connected_fraction_x/y/z`；
3. **真正的三维固体骨架特征**：当 `ENABLE_SOLID_SKELETON = True` 时，对 `solid_mask` 做三维骨架化，提取骨架体素数、端点、分叉点、连通分量、平均度、最大度和方向性骨架贯通等特征。

如果骨架化太慢，可以把：

```python
ENABLE_SOLID_SKELETON = False
```

这样仍会保留固体相连通特征和力学代理参数。


In [ ]:
# ============================================================
# 5. 力学代理参数、固体相连通特征与三维固体骨架特征
# ============================================================

def compute_pore_surface_area_density(pore_mask):
    face_area = {
        0: VX_UM * VY_UM,
        1: VX_UM * VZ_UM,
        2: VY_UM * VZ_UM,
    }

    area_um2 = 0.0

    for axis in [0, 1, 2]:
        sl1 = [slice(None)] * 3
        sl2 = [slice(None)] * 3

        sl1[axis] = slice(0, -1)
        sl2[axis] = slice(1, None)

        a = pore_mask[tuple(sl1)]
        b = pore_mask[tuple(sl2)]

        interface = a != b
        area_um2 += np.sum(interface) * face_area[axis]

    volume_um3 = (
        pore_mask.shape[0] * VZ_UM
        * pore_mask.shape[1] * VY_UM
        * pore_mask.shape[2] * VX_UM
    )

    density = area_um2 / max(volume_um3, 1e-30)
    return float(area_um2), float(density)


def downsample_binary_mask(binary_mask, max_size=100, threshold=0.5):
    """
    对三维二值 mask 做可选降采样。
    当 max_dim <= max_size 时不降采样。
    """
    max_dim = max(binary_mask.shape)

    if max_dim <= max_size:
        return binary_mask.astype(bool), 1

    factor = int(math.ceil(max_dim / max_size))
    new_shape = tuple((s // factor) * factor for s in binary_mask.shape)

    cropped = binary_mask[
        0:new_shape[0],
        0:new_shape[1],
        0:new_shape[2]
    ]

    reduced = block_reduce(
        cropped.astype(np.float32),
        block_size=(factor, factor, factor),
        func=np.mean
    )

    out = reduced >= threshold
    return out.astype(bool), factor



def downsample_solid_mask_for_mechanics(solid_mask, max_size=64, threshold=None):
    if threshold is None:
        threshold = MECH_SOLID_THRESHOLD

    return downsample_binary_mask(
        solid_mask,
        max_size=max_size,
        threshold=threshold
    )


def _solve_scalar_spring_stiffness_once(solid_mask, axis, max_size, threshold):
    """
    单次弹簧网络求解。

    关键修复：
    1. 求解失败返回 np.nan + status，而不是 0；
    2. 反力计算统计 inlet 与非 inlet 的所有连接边，
       不再只统计轴向下一层边，避免弯曲/曲折承载路径被误判为 0；
    3. 用自由节点/固定节点分块求解，降低 Dirichlet 边界处理导致的奇异风险。
    """
    result = {
        "E": np.nan,
        "status": "unknown",
        "n_nodes": 0,
        "n_edges": 0,
        "n_inlet_nodes": 0,
        "n_outlet_nodes": 0,
        "reaction": np.nan,
        "max_size_used": max_size,
        "threshold_used": threshold,
    }

    try:
        solid_ds, factor = downsample_solid_mask_for_mechanics(
            solid_mask,
            max_size=max_size,
            threshold=threshold
        )
    except Exception as e:
        result["status"] = f"downsample_failed: {e}"
        return result

    solid_perc, is_perc, _ = get_axis_percolating_mask(solid_ds, axis=axis)

    if not is_perc or solid_perc.sum() == 0:
        result["E"] = 0.0
        result["status"] = "not_percolating"
        return result

    coords = np.argwhere(solid_perc)
    n_nodes = coords.shape[0]
    result["n_nodes"] = int(n_nodes)

    if n_nodes < 2:
        result["status"] = "insufficient_nodes"
        return result

    shape = solid_perc.shape
    id_map = -np.ones(shape, dtype=np.int32)
    id_map[solid_perc] = np.arange(n_nodes, dtype=np.int32)

    inlet = coords[:, axis] == 0
    outlet = coords[:, axis] == (shape[axis] - 1)

    result["n_inlet_nodes"] = int(inlet.sum())
    result["n_outlet_nodes"] = int(outlet.sum())

    if inlet.sum() == 0 or outlet.sum() == 0:
        result["status"] = "boundary_nodes_missing"
        return result

    # 使用 6 邻域构建标量弹簧网络。只记录正方向，避免重复边。
    neighbor_shifts = [
        (1, 0, 0),
        (0, 1, 0),
        (0, 0, 1),
    ]

    edge_i = []
    edge_j = []

    for i, (z, y, x) in enumerate(coords):
        for dz, dy, dx in neighbor_shifts:
            nz = z + dz
            ny = y + dy
            nx = x + dx

            if nz < 0 or nz >= shape[0]:
                continue
            if ny < 0 or ny >= shape[1]:
                continue
            if nx < 0 or nx >= shape[2]:
                continue

            j = id_map[nz, ny, nx]
            if j >= 0:
                edge_i.append(i)
                edge_j.append(int(j))

    n_edges = len(edge_i)
    result["n_edges"] = int(n_edges)

    if n_edges == 0:
        result["status"] = "no_spring_edges"
        return result

    edge_i = np.asarray(edge_i, dtype=np.int64)
    edge_j = np.asarray(edge_j, dtype=np.int64)
    edge_g = np.ones(n_edges, dtype=np.float64)

    rows = np.concatenate([edge_i, edge_i, edge_j, edge_j])
    cols = np.concatenate([edge_i, edge_j, edge_i, edge_j])
    data = np.concatenate([edge_g, -edge_g, -edge_g, edge_g])

    Lmat = sparse.csr_matrix((data, (rows, cols)), shape=(n_nodes, n_nodes))

    fixed = inlet | outlet
    u = np.zeros(n_nodes, dtype=np.float64)
    u[inlet] = 1.0
    u[outlet] = 0.0

    free = ~fixed

    try:
        if np.any(free):
            A_ff = Lmat[free][:, free]
            b_f = -Lmat[free][:, fixed].dot(u[fixed])

            try:
                u_free, info = cg(A_ff, b_f, rtol=1e-8, maxiter=3000)
                if info != 0 or not np.all(np.isfinite(u_free)):
                    u_free = spsolve(A_ff, b_f)
            except Exception:
                u_free = spsolve(A_ff, b_f)

            if not np.all(np.isfinite(u_free)):
                result["status"] = "displacement_solution_nonfinite"
                return result

            u[free] = u_free
    except Exception as e:
        result["status"] = f"linear_solve_failed: {e}"
        return result

    # 反力：统计所有 inlet 节点与非 inlet 节点之间的弹簧通量。
    # 旧代码只统计轴向下一层边，曲折路径很容易被误判为 reaction=0。
    reaction = 0.0
    for i, j in zip(edge_i, edge_j):
        i = int(i)
        j = int(j)

        if inlet[i] and not inlet[j]:
            reaction += max(u[i] - u[j], 0.0)
        elif inlet[j] and not inlet[i]:
            reaction += max(u[j] - u[i], 0.0)

    result["reaction"] = float(reaction)

    if reaction <= 0 or not np.isfinite(reaction):
        result["status"] = "zero_or_nonfinite_reaction"
        return result

    L = max(shape[axis] - 1, 1)

    other_axes = [0, 1, 2]
    other_axes.remove(axis)
    area = shape[other_axes[0]] * shape[other_axes[1]]

    normalized_stiffness = reaction * L / max(area, 1e-30)
    E_eff = MATRIX_E_GPA * normalized_stiffness

    if not np.isfinite(E_eff) or E_eff <= 0:
        result["status"] = "invalid_E"
        return result

    result["E"] = float(E_eff)
    result["status"] = "ok"
    return result


def solve_scalar_spring_stiffness(solid_mask, axis):
    """
    稳健版力学代理求解。

    优先使用降采样网络；如果出现异常 0 或求解失败，则回退到原分辨率/更大尺寸重算。
    只有固相真实不贯通时才返回 E=0；其他失败均返回 NaN 和明确状态。
    """
    max_dim = max(solid_mask.shape)

    attempts = [
        ("downsampled", MECH_MAX_SIZE, MECH_SOLID_THRESHOLD),
    ]

    # BLOCK_SIZE=100 时，回退到原分辨率代价可以接受，可显著减少下采样造成的假失败。
    if max_dim > MECH_MAX_SIZE:
        attempts.append(("full_resolution", max_dim, MECH_SOLID_THRESHOLD))

    last = None

    for attempt_name, max_size, threshold in attempts:
        res = _solve_scalar_spring_stiffness_once(
            solid_mask=solid_mask,
            axis=axis,
            max_size=max_size,
            threshold=threshold
        )
        res["attempt"] = attempt_name
        last = res

        if res["status"] == "ok":
            return res

        # 如果降采样后显示不贯通，仍然尝试原分辨率；
        # 如果原分辨率也不贯通，再把 0 视为物理意义上的 0。
        if attempt_name == "full_resolution" and res["status"] == "not_percolating":
            return res

    if last is None:
        return {
            "E": np.nan,
            "status": "no_attempt",
            "attempt": "",
            "n_nodes": 0,
            "n_edges": 0,
            "n_inlet_nodes": 0,
            "n_outlet_nodes": 0,
            "reaction": np.nan,
            "max_size_used": np.nan,
            "threshold_used": np.nan,
        }

    # 到这里说明不是可靠的物理 0，而是求解异常；强制把 E 改成 NaN，避免污染训练标签。
    if last.get("status") != "not_percolating":
        last["E"] = np.nan

    return last


def compute_solid_directional_features(solid_mask):
    """
    固体相方向性连通特征。
    注意：这是固体相连通性，不是骨架化结果。
    """
    result = {}

    for direction in ["x", "y", "z"]:
        axis = DIRECTION_AXIS_MASK[direction]
        solid_perc, is_perc, ncomp = get_axis_percolating_mask(solid_mask, axis=axis)

        result[f"solid_percolates_{direction}"] = int(is_perc)
        result[f"solid_percolating_components_{direction}"] = int(ncomp)
        result[f"solid_connected_fraction_{direction}"] = float(np.mean(solid_perc))

    return result


def build_3d_neighbor_kernel():
    """
    26 邻域卷积核，用于统计骨架体素的邻居数量。
    """
    kernel = np.ones((3, 3, 3), dtype=np.uint8)
    kernel[1, 1, 1] = 0
    return kernel


def compute_skeleton_directional_features(skeleton_mask):
    """
    判断骨架是否在 X/Y/Z 方向贯通。
    """
    result = {}

    for direction in ["x", "y", "z"]:
        axis = DIRECTION_AXIS_MASK[direction]
        skel_perc, is_perc, ncomp = get_axis_percolating_mask(skeleton_mask, axis=axis)

        result[f"solid_skeleton_percolates_{direction}"] = int(is_perc)
        result[f"solid_skeleton_percolating_components_{direction}"] = int(ncomp)
        result[f"solid_skeleton_connected_fraction_{direction}"] = float(np.mean(skel_perc))

    return result


def compute_solid_skeleton_features(solid_mask):
    """
    从固体相 mask 中提取真正的三维骨架特征。

    流程：
        solid_mask -> skeletonize_3d -> skeleton features

    注意：
        骨架化对噪声和分割阈值敏感。
        如果速度过慢，可设置 ENABLE_SOLID_SKELETON=False。
    """
    result = {
        "solid_skeleton_enabled": int(ENABLE_SOLID_SKELETON),
        "solid_skeleton_voxel_count": np.nan,
        "solid_skeleton_volume_fraction": np.nan,
        "solid_skeleton_length_um_approx": np.nan,
        "solid_skeleton_component_count": np.nan,
        "solid_skeleton_endpoint_count": np.nan,
        "solid_skeleton_junction_count": np.nan,
        "solid_skeleton_branch_voxel_count": np.nan,
        "solid_skeleton_avg_degree": np.nan,
        "solid_skeleton_max_degree": np.nan,
        "solid_skeleton_downsample_factor": np.nan,
        "solid_skeleton_status": "disabled",
    }

    for direction in ["x", "y", "z"]:
        result[f"solid_skeleton_percolates_{direction}"] = np.nan
        result[f"solid_skeleton_percolating_components_{direction}"] = np.nan
        result[f"solid_skeleton_connected_fraction_{direction}"] = np.nan

    if not ENABLE_SOLID_SKELETON:
        return result

    try:
        solid_for_skeleton, factor = downsample_binary_mask(
            solid_mask,
            max_size=SKELETON_MAX_SIZE,
            threshold=MECH_SOLID_THRESHOLD
        )

        if solid_for_skeleton.sum() == 0:
            result["solid_skeleton_status"] = "empty_solid"
            result["solid_skeleton_downsample_factor"] = factor
            return result

        skeleton = skeletonize_3d_compatible(solid_for_skeleton).astype(bool)

        voxel_count = int(skeleton.sum())
        result["solid_skeleton_voxel_count"] = voxel_count
        result["solid_skeleton_volume_fraction"] = float(np.mean(skeleton))
        result["solid_skeleton_downsample_factor"] = int(factor)

        # 如果降采样了，每个骨架体素代表 factor 倍体素尺度，这里给近似长度。
        result["solid_skeleton_length_um_approx"] = float(voxel_count * VOXEL_SIZE_UM * factor)

        if voxel_count == 0:
            result["solid_skeleton_status"] = "empty_skeleton"
            return result

        labels, ncomp = ndi.label(
            skeleton,
            structure=np.ones((3, 3, 3), dtype=bool)
        )
        result["solid_skeleton_component_count"] = int(ncomp)

        kernel = build_3d_neighbor_kernel()
        degree_map = ndi.convolve(
            skeleton.astype(np.uint8),
            kernel,
            mode="constant",
            cval=0
        )

        degree_values = degree_map[skeleton].astype(float)

        result["solid_skeleton_endpoint_count"] = int(np.sum(degree_values == 1))
        result["solid_skeleton_junction_count"] = int(np.sum(degree_values >= 3))
        result["solid_skeleton_branch_voxel_count"] = int(np.sum(degree_values == 2))
        result["solid_skeleton_avg_degree"] = float(np.mean(degree_values))
        result["solid_skeleton_max_degree"] = float(np.max(degree_values))

        result.update(compute_skeleton_directional_features(skeleton))
        result["solid_skeleton_status"] = "ok"

    except Exception as e:
        result["solid_skeleton_status"] = f"failed: {e}"

    return result


def compute_mechanics_features_v2(pore_mask, sample_id, origin_zyx, solid_mask=None):
    """
    计算力学代理标签、固体相方向性连通特征和可选三维固体骨架特征。

    参数：
        pore_mask:
            孔隙 mask。True 表示孔隙。
        solid_mask:
            固体相 mask。True 表示固体。
            推荐由原始灰度块计算：solid_mask = block > PORE_MAX。
            如果未提供，则使用 ~pore_mask 作为退化方案。
    """
    porosity = float(np.mean(pore_mask))

    if solid_mask is None:
        solid_mask = ~pore_mask
        solid_threshold_method = "solid_mask_from_inverse_pore_mask"
    else:
        solid_mask = solid_mask.astype(bool)
        solid_threshold_method = f"solid_mask_from_raw_gray_gt_{PORE_MAX}"

    solid_fraction = float(np.mean(solid_mask))

    pore_area_um2, pore_area_density = compute_pore_surface_area_density(pore_mask)

    solid_features = compute_solid_directional_features(solid_mask)
    skeleton_features = compute_solid_skeleton_features(solid_mask)

    # Python 数组顺序：Z, Y, X
    Ez_res = solve_scalar_spring_stiffness(solid_mask, axis=0)
    Ey_res = solve_scalar_spring_stiffness(solid_mask, axis=1)
    Ex_res = solve_scalar_spring_stiffness(solid_mask, axis=2)

    Ex = float(Ex_res.get("E", np.nan))
    Ey = float(Ey_res.get("E", np.nan))
    Ez = float(Ez_res.get("E", np.nan))

    E_values = np.array([Ex, Ey, Ez], dtype=np.float64)
    finite_positive_E = E_values[np.isfinite(E_values) & (E_values > 0)]

    if finite_positive_E.size > 0:
        E_mean = float(np.mean(finite_positive_E))
    else:
        E_mean = np.nan

    if np.all(np.isfinite(E_values)) and np.all(E_values > 0):
        E_geom = float((Ex * Ey * Ez) ** (1.0 / 3.0))
        E_anisotropy = float(np.max(E_values) / max(np.min(E_values), 1e-30))
    else:
        E_geom = np.nan
        E_anisotropy = np.nan

    nu_eff = MATRIX_NU * solid_fraction
    nu_eff = float(np.clip(nu_eff, 0.01, 0.45))

    if np.isfinite(E_geom) and E_geom > 0:
        K_eff = E_geom / (3.0 * (1.0 - 2.0 * nu_eff))
        G_eff = E_geom / (2.0 * (1.0 + nu_eff))
    else:
        K_eff = np.nan
        G_eff = np.nan

    z0, y0, x0 = origin_zyx

    row = {
        "sample_id": sample_id,
        "z_start": z0,
        "y_start": y0,
        "x_start": x0,
        "block_size": BLOCK_SIZE,
        "voxel_size_um": VOXEL_SIZE_UM,

        # 阈值说明
        "pore_gray_min": PORE_MIN,
        "pore_gray_max": PORE_MAX,
        "solid_gray_min": SOLID_MIN,
        "solid_gray_max": SOLID_MAX,
        "solid_threshold_method": solid_threshold_method,

        # 基础相分数
        "porosity": porosity,
        "solid_fraction": solid_fraction,

        # 表面积密度
        "pore_surface_area_um2": pore_area_um2,
        "pore_surface_area_density_um2_per_um3": pore_area_density,

        # 力学代理标签
        "matrix_E_GPa": MATRIX_E_GPA,
        "matrix_nu": MATRIX_NU,
        "Ex_spring_GPa": Ex,
        "Ey_spring_GPa": Ey,
        "Ez_spring_GPa": Ez,
        "Ex_status": Ex_res.get("status", ""),
        "Ey_status": Ey_res.get("status", ""),
        "Ez_status": Ez_res.get("status", ""),
        "Ex_solver_attempt": Ex_res.get("attempt", ""),
        "Ey_solver_attempt": Ey_res.get("attempt", ""),
        "Ez_solver_attempt": Ez_res.get("attempt", ""),
        "Ex_reaction": Ex_res.get("reaction", np.nan),
        "Ey_reaction": Ey_res.get("reaction", np.nan),
        "Ez_reaction": Ez_res.get("reaction", np.nan),
        "Ex_n_nodes": Ex_res.get("n_nodes", np.nan),
        "Ey_n_nodes": Ey_res.get("n_nodes", np.nan),
        "Ez_n_nodes": Ez_res.get("n_nodes", np.nan),
        "Ex_n_edges": Ex_res.get("n_edges", np.nan),
        "Ey_n_edges": Ey_res.get("n_edges", np.nan),
        "Ez_n_edges": Ez_res.get("n_edges", np.nan),
        "E_mean_GPa": E_mean,
        "E_geom_mean_GPa": E_geom,
        "E_anisotropy_max_min": E_anisotropy,
        "nu_eff_estimated": nu_eff,
        "K_eff_estimated_GPa": K_eff,
        "G_eff_estimated_GPa": G_eff,
        "mechanics_method": (
            f"solid_threshold_gt_{PORE_MAX}_"
            f"downsampled_scalar_spring_network_max_size_{MECH_MAX_SIZE}_"
            f"skeleton_enabled_{ENABLE_SOLID_SKELETON}"
        ),
    }

    row.update(solid_features)
    row.update(skeleton_features)
    return row


def write_mechanics_xlsx(mech_row, out_path: Path):
    df = pd.DataFrame([mech_row])
    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name="Mechanics", index=False)


In [ ]:
# ============================================================
# 5.1 运行力学代理参数计算
# ============================================================

def process_mechanics_row(row_dict):
    """
    单个样本的力学代理参数计算函数。
    设计成独立函数，方便 joblib 并行。
    """
    row = pd.Series(row_dict)

    sample_id = row["sample_id"]
    origin_zyx = (
        int(row.get("z_start", 0)),
        int(row.get("y_start", 0)),
        int(row.get("x_start", 0)),
    )

    pore_file = Path(row["pore_file"])
    mech_xlsx = MECH_OUT / f"{sample_id}.Mechanics_v2.xlsx"

    try:
        if not pore_file.exists():
            raise FileNotFoundError(f"找不到 pore mask: {pore_file}")

        pore_mask = np.load(pore_file).astype(bool)

        # 固体相建议直接从原始灰度块阈值得到：
        #   pore  = 0–18
        #   solid = 19–255，也就是 block > PORE_MAX
        # 如果 raw_file 不存在，则退回使用 ~pore_mask。
        solid_mask = None
        raw_file = Path(row.get("raw_file", ""))

        if raw_file.exists():
            raw_block = read_raw_block(raw_file)
            solid_mask = (raw_block > PORE_MAX) & (raw_block <= SOLID_MAX)

        mech_row = compute_mechanics_features_v2(
            pore_mask=pore_mask,
            solid_mask=solid_mask,
            sample_id=sample_id,
            origin_zyx=origin_zyx
        )

        # 逐样本 Excel 写入非常慢，默认关闭。
        if SAVE_INDIVIDUAL_MECH_XLSX:
            write_mechanics_xlsx(mech_row, mech_xlsx)

        dir_statuses = [
            mech_row.get("Ex_status", ""),
            mech_row.get("Ey_status", ""),
            mech_row.get("Ez_status", ""),
        ]
        if all(s == "ok" for s in dir_statuses):
            status = "ok"
        elif any(str(s).startswith("not_percolating") for s in dir_statuses):
            status = "ok_with_nonpercolating_solid_direction"
        else:
            status = "ok_with_directional_solver_warning"

    except Exception as e:
        mech_row = {
            "sample_id": sample_id,
            "z_start": origin_zyx[0],
            "y_start": origin_zyx[1],
            "x_start": origin_zyx[2],
            "block_size": BLOCK_SIZE,
        }
        status = f"failed: {e}"

    mech_row["mechanics_xlsx"] = str(mech_xlsx) if SAVE_INDIVIDUAL_MECH_XLSX else ""
    mech_row["mechanics_status"] = status

    return mech_row


def compute_mechanics_dataset_v2():
    pore_summary_csv = SUMMARY_OUT / f"{DATASET_PREFIX}_pore_summary_v2.csv"
    if not pore_summary_csv.exists():
        raise FileNotFoundError("请先运行 Step 2：孔隙分割。")

    df = pd.read_csv(pore_summary_csv)

    print("=" * 90)
    print("Step 4: 力学代理参数、固体相连通特征与三维固体骨架特征计算开始")
    print(f"样本数量: {len(df)}")
    print(f"Mechanics 输出目录: {MECH_OUT}")
    print(f"是否并行: {PARALLEL_MECHANICS}, 并行任务数: {N_JOBS_MECHANICS}")
    print(f"是否保存逐样本 Mechanics.xlsx: {SAVE_INDIVIDUAL_MECH_XLSX}")
    print(f"是否启用三维固体骨架化: {ENABLE_SOLID_SKELETON}")
    print(f"骨架化最大尺寸: {SKELETON_MAX_SIZE}")
    print("=" * 90)

    rows = df.to_dict("records")

    if PARALLEL_MECHANICS and len(rows) > 1:
        with tqdm_joblib(tqdm(total=len(rows), desc="Step 4 力学参数计算进度", unit="样本")):
            summary_rows = Parallel(
                n_jobs=N_JOBS_MECHANICS,
                backend="loky",
                batch_size=1,
                verbose=0
            )(
                delayed(process_mechanics_row)(r)
                for r in rows
            )
    else:
        summary_rows = []
        for r in tqdm(rows, desc="Step 4 力学参数计算进度", unit="样本"):
            summary_rows.append(process_mechanics_row(r))

    summary_df = pd.DataFrame(summary_rows)

    summary_csv = SUMMARY_OUT / f"{DATASET_PREFIX}_mechanics_summary_v2.csv"
    summary_xlsx = SUMMARY_OUT / f"{DATASET_PREFIX}_mechanics_summary_v2.xlsx"

    # summary 目录保存总表
    summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")
    summary_df.to_excel(summary_xlsx, index=False)

    # Mechanics 目录也保存一份总表，方便查看。
    mech_summary_csv = MECH_OUT / f"{DATASET_PREFIX}_mechanics_summary_v2.csv"
    mech_summary_xlsx = MECH_OUT / f"{DATASET_PREFIX}_mechanics_summary_v2.xlsx"
    summary_df.to_csv(mech_summary_csv, index=False, encoding="utf-8-sig")
    summary_df.to_excel(mech_summary_xlsx, index=False)

    print("=" * 90)
    print("Step 4 完成：力学代理参数、固体相连通特征与三维固体骨架特征计算完成")
    print(f"汇总 CSV: {summary_csv}")
    print(f"汇总 Excel: {summary_xlsx}")
    print(f"Mechanics 文件夹总表: {mech_summary_csv}")
    print("=" * 90)

    return summary_df


# 运行 Step 4
mechanics_summary_df = compute_mechanics_dataset_v2()
mechanics_summary_df.head()


## 6. Step 5：合并最终 GNN 数据集总表

最终生成：

```text
Berea_GNN_dataset_v2_master.csv
Berea_GNN_dataset_v2_master.xlsx
```

这个总表将包含：

- 空间索引与空间划分字段；
- 孔隙率与方向性连通孔隙率；
- 方向性贯通标签；
- 方向性渗透率标签；
- 图级拓扑特征；
- 固体骨架特征；
- 力学代理标签。

In [ ]:
# ============================================================
# 6. 合并最终数据集总表
# ============================================================

def merge_master_dataset_v2():
    pore_csv = SUMMARY_OUT / f"{DATASET_PREFIX}_pore_summary_v2.csv"
    perm_csv = SUMMARY_OUT / f"{DATASET_PREFIX}_permeation_summary_v2.csv"
    mech_csv = SUMMARY_OUT / f"{DATASET_PREFIX}_mechanics_summary_v2.csv"

    if not pore_csv.exists() or not perm_csv.exists() or not mech_csv.exists():
        raise FileNotFoundError("请先完整运行 Step 2、Step 3 和 Step 4。")

    pore_df = pd.read_csv(pore_csv)
    perm_df = pd.read_csv(perm_csv)
    mech_df = pd.read_csv(mech_csv)

    master = pore_df.merge(
        perm_df,
        on=[c for c in ["sample_id", "z_start", "y_start", "x_start", "block_size"] if c in pore_df.columns],
        how="left",
        suffixes=("", "_perm")
    ).merge(
        mech_df,
        on=[c for c in ["sample_id", "z_start", "y_start", "x_start", "block_size"] if c in pore_df.columns],
        how="left",
        suffixes=("", "_mech")
    )

    master_csv = SUMMARY_OUT / f"{DATASET_PREFIX}_GNN_dataset_v2_master.csv"
    master_xlsx = SUMMARY_OUT / f"{DATASET_PREFIX}_GNN_dataset_v2_master.xlsx"

    master.to_csv(master_csv, index=False, encoding="utf-8-sig")
    master.to_excel(master_xlsx, index=False)

    print("=" * 90)
    print("最终 GNN 数据集总表已生成")
    print(f"CSV: {master_csv}")
    print(f"Excel: {master_xlsx}")
    print(f"样本数量: {len(master)}")
    print("=" * 90)

    return master


master_df = merge_master_dataset_v2()
master_df.head()


## 7. Step 6：数据集质量检查

这一部分用于快速判断新数据集是否比旧数据集更适合训练 GNN。

In [ ]:
# ============================================================
# 7. 数据集质量检查
# ============================================================

def dataset_quality_report(master_df):
    print("=" * 90)
    print("1. 样本数量")
    print("=" * 90)
    print("总样本数：", len(master_df))

    print("\n" + "=" * 90)
    print("2. 孔隙方向性贯通比例")
    print("=" * 90)
    for d in ["x", "y", "z"]:
        col = f"percolates_{d}"
        if col in master_df.columns:
            print(f"{col}: {master_df[col].mean():.4f}")

    print("\n" + "=" * 90)
    print("3. 固体骨架方向性贯通比例")
    print("=" * 90)
    for d in ["x", "y", "z"]:
        col = f"solid_percolates_{d}"
        if col in master_df.columns:
            print(f"{col}: {master_df[col].mean():.4f}")

    print("\n" + "=" * 90)
    print("3. 固体骨架方向性贯通比例")
    print("=" * 90)
    for d in ["x", "y", "z"]:
        col = f"solid_skeleton_percolates_{d}"
        if col in master_df.columns:
            print(f"{col}: {master_df[col].mean():.4f}")
    if "solid_skeleton_status" in master_df.columns:
        print("\n骨架化状态：")
        print(master_df["solid_skeleton_status"].value_counts(dropna=False))

    print("\n" + "=" * 90)
    print("4. 渗透率描述统计")
    print("=" * 90)
    k_cols = ["kx_um2", "ky_um2", "kz_um2"]
    display(master_df[k_cols].describe())

    print("\n渗透率方向状态统计：")
    for d in ["x", "y", "z"]:
        col = f"k{d}_status"
        if col in master_df.columns:
            print(f"\n{col}:")
            print(master_df[col].value_counts(dropna=False).head(20))


    print("\n" + "=" * 90)
    print("5. 力学参数描述统计")
    print("=" * 90)
    e_cols = ["Ex_spring_GPa", "Ey_spring_GPa", "Ez_spring_GPa"]
    display(master_df[e_cols].describe())

    print("\n力学方向状态统计：")
    for d in ["x", "y", "z"]:
        col = f"E{d}_status"
        if col in master_df.columns:
            print(f"\n{col}:")
            print(master_df[col].value_counts(dropna=False).head(20))

    print("\n异常 0 / NaN 快速检查：")
    for col in k_cols + e_cols:
        if col in master_df.columns:
            zero_count = int((pd.to_numeric(master_df[col], errors="coerce") == 0).sum())
            nan_count = int(pd.to_numeric(master_df[col], errors="coerce").isna().sum())
            print(f"{col}: zero={zero_count}, NaN={nan_count}")


    print("\n" + "=" * 90)
    print("6. 空间划分样本数量")
    print("=" * 90)
    if "split_by_x" in master_df.columns:
        print("按 X 方向划分：")
        print(master_df["split_by_x"].value_counts())
    if "split_by_z" in master_df.columns:
        print("\n按 Z 方向划分：")
        print(master_df["split_by_z"].value_counts())

    print("\n" + "=" * 90)
    print("7. 关键字段缺失值统计")
    print("=" * 90)
    key_cols = [
        "porosity",
        "connected_porosity_x", "connected_porosity_y", "connected_porosity_z",
        "percolates_x", "percolates_y", "percolates_z",
        "kx_um2", "ky_um2", "kz_um2",
        "solid_fraction",
        "solid_percolates_x", "solid_percolates_y", "solid_percolates_z",
        "solid_connected_fraction_x", "solid_connected_fraction_y", "solid_connected_fraction_z",
        "solid_skeleton_status",
        "solid_skeleton_voxel_count",
        "solid_skeleton_endpoint_count",
        "solid_skeleton_junction_count",
        "solid_skeleton_component_count",
        "solid_skeleton_percolates_x", "solid_skeleton_percolates_y", "solid_skeleton_percolates_z",
        "Ex_spring_GPa", "Ey_spring_GPa", "Ez_spring_GPa",
        "avg_pore_radius_cm", "avg_throat_radius_cm",
        "avg_betweenness", "avg_closeness", "avg_pagerank"
    ]
    key_cols = [c for c in key_cols if c in master_df.columns]
    display(master_df[key_cols].isna().sum())


dataset_quality_report(master_df)


In [ ]:
# ============================================================
# 8. 8019样本完整性检查
# ============================================================

expected_blocks_per_rock = len(build_positions(SHAPE_ZYX, BLOCK_SIZE, STRIDE, None))
expected_total_blocks = len(ROCK_SAMPLES) * expected_blocks_per_rock

print("每个岩样理论块数:", expected_blocks_per_rock)
print("理论总块数:", expected_total_blocks)
print("当前 master 样本数:", len(master_df))

if MAX_BLOCKS_PER_ROCK is None:
    if len(master_df) != expected_total_blocks:
        print("警告：当前 master 样本数与理论完整样本数不一致，请检查中间步骤是否有失败。")
    else:
        print("完整性检查通过：已生成完整 11×729 = 8019 样本。")
else:
    print("当前仍处于测试模式：MAX_BLOCKS_PER_ROCK =", MAX_BLOCKS_PER_ROCK)
    print("正式重建完整数据集请改为：MAX_BLOCKS_PER_ROCK = None")


## 8. 后续训练建议

基于这个 v2 数据集，后续训练 GNN 时建议采用以下方式：

### 渗流模型

不要直接对所有样本回归 `kx/ky/kz`。推荐二阶段建模：

1. **方向性贯通分类**
   - 输入：方向性 PNM 图或 3D 体素；
   - 输出：`percolates_x/y/z`。

2. **贯通样本渗透率回归**
   - 只对 `percolates_direction = 1` 的样本训练回归模型；
   - 标签：`log10(k_direction + EPS)`。

### 力学模型

力学模型最好不要只用孔隙 PNM 图。建议至少加入：

- `solid_fraction`
- `solid_percolates_x/y/z`
- `solid_connected_fraction_x/y/z`
- `pore_surface_area_density`

如果后续要进一步提高效果，可以构建固体骨架图或引入 3D CNN 体素分支。

### 数据划分

论文中建议使用空间划分：

```text
split_by_x 或 split_by_z
```

而不是随机划分，避免 100×100×100 块在 stride=50 时高度重叠造成数据泄露。


## 后续训练提醒

PM-GNN训练 Notebook 默认读取：

```text
H:\Deepl_Project\GNN\ROCK_data\data\DRP317_11sandstone_GNN_dataset_v3\summary\DRP317_11sandstone_GNN_dataset_v2_master.csv
```

本 Notebook 已按这个路径和文件名生成总表。

正式重建完整数据集前，请确认：

```python
MAX_BLOCKS_PER_ROCK = None
STRIDE = 100
BLOCK_SIZE = 200
```

完整数据量为：

```text
11 × 729 = 8019
```

如果只是测试流程，可以保留：

```python
MAX_BLOCKS_PER_ROCK = 10
```
